# Notebook E — Generalization Testing

This notebook tests whether the classifiers trained in notebooks 03a/03b generalize beyond their training conditions. Two types of generalization are evaluated:

1. **Cross-cell-line**: a model trained on MCF7 cells is applied to HCC1806 cells (and vice versa). Same technology (DropSeq), different biology.
2. **Cross-technology**: a model trained on DropSeq data is applied to SmartSeq data from the same cell line. Same biology, different sequencing platform.

For each scenario we compare cross-condition accuracy to in-distribution performance and report the drop.

In [ ]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

sns.set_theme(style='whitegrid')

MODELS_DIR    = '../outputs/models'
DATA_DROPSEQ  = '../data/DropSeq'
DATA_SMARTSEQ = '../data/SmartSeq'

## 1. Load Models and Gene Lists

The `.pkl` models expect exactly the 500 genes selected during training (by mutual information on their own training data). We reload those gene lists from the CSVs saved by notebooks 03a/03b.

In [ ]:
def load_model(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

models = {
    'MCF7': {
        'LogisticRegression': load_model(f'{MODELS_DIR}/MCF7_DropSeq_LogisticRegression.pkl'),
        'SVM':                load_model(f'{MODELS_DIR}/MCF7_DropSeq_SVM.pkl'),
        'RandomForest':       load_model(f'{MODELS_DIR}/MCF7_DropSeq_RandomForest.pkl'),
        'GradientBoosting':   load_model(f'{MODELS_DIR}/MCF7_DropSeq_GradientBoosting.pkl'),
    },
    'HCC1806': {
        'LogisticRegression': load_model(f'{MODELS_DIR}/HCC1806_DropSeq_Logistic Regression.pkl'),
        'SVM':                load_model(f'{MODELS_DIR}/HCC1806_DropSeq_SVM.pkl'),
        'RandomForest':       load_model(f'{MODELS_DIR}/HCC1806_DropSeq_Random Forest.pkl'),
        'GradientBoosting':   load_model(f'{MODELS_DIR}/HCC1806_DropSeq_Gradient Boosting.pkl'),
    },
}

mcf7_genes = pd.read_csv('../outputs/MCF7_top_genes.csv',    index_col=0).index.tolist()
hcc_genes  = pd.read_csv('../outputs/HCC1806_top_genes.csv', index_col=0).index.tolist()

overlap = set(mcf7_genes) & set(hcc_genes)
print(f'MCF7 training genes:    {len(mcf7_genes)}')
print(f'HCC1806 training genes: {len(hcc_genes)}')
print(f'Genes in common:        {len(overlap)}')
print()
print('Models loaded:', {k: list(v.keys()) for k, v in models.items()})

## 2. Helper Functions

In [ ]:
def load_dropseq(cell_line):
    """Load DropSeq train matrix. Returns DataFrame (cells x genes) and label array."""
    path = f'{DATA_DROPSEQ}/{cell_line}_Filtered_Normalised_3000_Data_train.txt'
    raw = pd.read_csv(path, sep=' ', index_col=0).T
    y = raw.index.str.split('_').str[-1].values
    return raw, y


def load_smartseq(cell_line):
    """Load SmartSeq train matrix and extract Hypoxia/Normoxia labels from filenames."""
    path = f'{DATA_SMARTSEQ}/{cell_line}_SmartS_Filtered_Normalised_3000_Data_train.txt'
    raw = pd.read_csv(path, sep=' ', index_col=0).T
    pat = re.compile(r'Hypoxia|Normoxia|Hypo|Norm', re.IGNORECASE)
    labels = []
    for name in raw.index:
        m = pat.search(name)
        if m:
            labels.append('Hypoxia' if m.group(0).lower().startswith('hyp') else 'Normoxia')
        else:
            labels.append('Unknown')
    y = np.array(labels)
    mask = y != 'Unknown'
    return raw[mask], y[mask]


def select_genes(df, gene_list):
    """
    Subset df to the training gene list, in training order.
    Genes absent from df (dropout / different platform) are filled with 0.
    """
    present = [g for g in gene_list if g in df.columns]
    missing = [g for g in gene_list if g not in df.columns]
    # build in one concat to avoid DataFrame fragmentation
    parts = [df[present]]
    if missing:
        parts.append(pd.DataFrame(0.0, index=df.index, columns=missing))
    full = pd.concat(parts, axis=1)
    return full[gene_list].values


def run_models(cell_line_models, X, y, scenario, train_cell, test_cell):
    rows = []
    for mname, model in cell_line_models.items():
        yp = model.predict(X)
        rows.append({
            'scenario':  scenario,
            'train':     train_cell,
            'test':      test_cell,
            'model':     mname,
            'acc':       round(accuracy_score(y, yp), 4),
            'bacc':      round(balanced_accuracy_score(y, yp), 4),
        })
    return rows

## 3. In-Distribution Baseline

We re-apply each model to the data it was **trained on**. This gives an upper-bound estimate of performance (the models have seen this data before, so scores will be inflated for models that overfit). It is included here as a reference point, not as a valid generalization estimate.

In [ ]:
results = []

for cell_line, gene_list in [('MCF7', mcf7_genes), ('HCC1806', hcc_genes)]:
    df, y = load_dropseq(cell_line)
    X = select_genes(df, gene_list)
    results += run_models(models[cell_line], X, y, 'in-dist', cell_line, cell_line)

pd.DataFrame(results).sort_values(['train', 'model'])

## 4. Cross-Cell-Line Generalization

MCF7-trained models are applied to HCC1806 data, and vice versa. Same technology (DropSeq), different cell line.

The MCF7 gene set and HCC1806 gene set share only 105 genes out of 500. For genes in the training set that are absent from the target cell line's data, we substitute 0 (not detected / below detection threshold).

In [ ]:
for train_cell, gene_list, test_cell in [
    ('MCF7',    mcf7_genes, 'HCC1806'),
    ('HCC1806', hcc_genes,  'MCF7'),
]:
    df, y = load_dropseq(test_cell)
    n_present = sum(1 for g in gene_list if g in df.columns)
    print(f'{train_cell} → {test_cell}: {n_present}/500 training genes present in target data')
    X = select_genes(df, gene_list)
    results += run_models(models[train_cell], X, y, 'cross-cell', train_cell, test_cell)

cross_cell_df = pd.DataFrame([r for r in results if r['scenario'] == 'cross-cell'])
cross_cell_df.sort_values(['train', 'model'])

## 5. Cross-Technology Generalization

DropSeq-trained models are applied to SmartSeq data from the **same** cell line. Same biology, different sequencing platform.

The 500 training genes were selected by mutual information on DropSeq data. The SmartSeq data has been filtered to the same 3 000-gene panel, but the DropSeq-specific top-500 overlap with this SmartSeq panel only partially — we report how many genes are actually available.

In [ ]:
for cell_line, gene_list in [('MCF7', mcf7_genes), ('HCC1806', hcc_genes)]:
    df, y = load_smartseq(cell_line)
    n_present = sum(1 for g in gene_list if g in df.columns)
    print(f'{cell_line} DropSeq → SmartSeq: {n_present}/500 training genes present, {len(y)} labeled cells')
    X = select_genes(df, gene_list)
    results += run_models(models[cell_line], X, y, 'cross-tech', cell_line, cell_line)

cross_tech_df = pd.DataFrame([r for r in results if r['scenario'] == 'cross-tech'])
cross_tech_df.sort_values(['train', 'model'])

## 6. Summary and Visualization

In [ ]:
results_df = pd.DataFrame(results)

# Average over train/test direction within each scenario
pivot = results_df.pivot_table(
    values='bacc',
    index='model',
    columns='scenario',
    aggfunc='mean',
)[['in-dist', 'cross-cell', 'cross-tech']]

fig, ax = plt.subplots(figsize=(9, 4))
palette = ['#2a9d8f', '#e76f51', '#e9c46a']
pivot.plot(kind='bar', ax=ax, color=palette, edgecolor='white', width=0.7)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Generalization: balanced accuracy by scenario and model')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.legend(title='Scenario')
ax.set_ylim(0, 1.08)
ax.axhline(0.5, linestyle='--', linewidth=0.8, color='grey', label='chance')
plt.tight_layout()
plt.savefig('../outputs/figures/generalization_summary.png', dpi=150)
plt.show()

print(pivot.round(3).to_string())

In [ ]:
# Performance drop relative to in-distribution
in_dist_mean = results_df[results_df['scenario'] == 'in-dist'].groupby('model')['bacc'].mean()
cross = results_df[results_df['scenario'] != 'in-dist'].copy()
cross['in_dist'] = cross['model'].map(in_dist_mean)
cross['drop']    = (cross['in_dist'] - cross['bacc']).round(3)

print('Performance drop (in-dist bacc − cross-condition bacc):')
print(cross[['scenario', 'train', 'test', 'model', 'bacc', 'drop']]
      .sort_values(['scenario', 'model']).to_string(index=False))

## 7. Interpretation

### In-distribution baseline

All four models score very high on their own training data (balanced accuracy 0.94–1.00). RandomForest achieves a perfect 1.00, which is a clear sign of overfitting to training samples rather than learning a generalizable signal. These in-distribution scores are not valid estimates of true generalization — they are included only as a reference ceiling.

### Cross-cell-line generalization

Performance drops substantially in both directions, but the drop is asymmetric:

- **MCF7 → HCC1806**: balanced accuracy falls to 0.54–0.57 across all models — barely above chance (0.50). The MCF7-selected genes explain very little of the hypoxia variation in HCC1806. This suggests the two cell lines express hypoxia very differently: the genes that are most informative in MCF7 are not the same ones that are informative in HCC1806.

- **HCC1806 → MCF7**: generalization is meaningfully better (0.64–0.80). SVM performs best at 0.80, LogisticRegression at 0.69. This asymmetry suggests that the HCC1806 hypoxia gene signature overlaps more with MCF7's response than the other way around — the HCC1806 genes capture a more universal hypoxia signal.

Only 179–212 of the 500 training genes are actually present in the target cell line's DropSeq data. The low gene overlap (the two cell lines were feature-selected independently) is a major contributor to the performance drop.

### Cross-technology generalization

This is the harder test, and the results reflect that:

- **Most models collapse to ~0.50 balanced accuracy** (random chance for a binary classification). GradientBoosting, RandomForest, and SVM all fail completely on both cell lines. This happens for a concrete structural reason: only 80–98 of the 500 DropSeq training genes are present in the SmartSeq data. The models were trained on a gene signature that does not exist — or was not measured — in the other technology. When ~84% of input features are zero-filled, the models have almost no real signal to work with.

- **LogisticRegression on MCF7 SmartSeq is a striking exception at 0.956 balanced accuracy.** This is unexpectedly high and likely reflects that Logistic Regression, being a simpler linear model with fewer parameters, happened to find a few of the 80 shared genes that carry a strong and consistent hypoxia signal across both technologies. This is not guaranteed to replicate: the SmartSeq MCF7 set has only 250 cells, so this score has high variance.

### Overall conclusion

The classifiers do not generalize robustly. Cross-cell-line performance is partial at best, and cross-technology performance mostly fails due to the small overlap between the feature sets selected independently for each condition. A model designed for generalization would need to:
1. Select features on a **shared gene panel** across both cell lines and technologies before training.
2. Be evaluated with held-out cross-validation, not by re-applying to training data.

The asymmetry in cross-cell generalization (HCC1806→MCF7 works better than MCF7→HCC1806) is biologically interesting and worth investigating: it suggests the HCC1806 hypoxia response involves genes that are more broadly hypoxia-relevant across cell types.